#  Arm Assembly Primer : Arithmetic
# 1. Integer arithmetic

In [ ]:
%%writefile arith.c
long add(long a, long b) { return a + b; }
long sub(long a, long b) { return a - b; }
long mul(long a, long b) { return a * b; }
long div_(long a, long b) { return a / b; }
long rem(long a, long b) { return a % b; }
long mul8(long a) { return a * 8; }
long axpb(long a, long x, long b) { return a * x + b; }
int add_int(int a, int b) { return a + b; }
unsigned long udiv_(unsigned long a, unsigned long b) { return a / b; }
long big_const(void) { return 1234567890123L; }

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S arith.c
grep -vE '^\s+\.|^\s*//|^\.Lfunc' arith.s

* You will find

<div>

| C | assembly | meaning |
|---|----------|---------|
| `a + b` | `add x0, x1, x0` | `x0 = x1 + x0` |
| `a - b` | `sub x0, x0, x1` | `x0 = x0 - x1` |
| `a * b` | `mul x0, x1, x0` | `x0 = x1 * x0` |
| `a / b` | `sdiv x0, x0, x1` | signed division |
| `a % b` | `sdiv x8, x0, x1` + `msub x0, x8, x1, x0` | `x8 = a / b`; `x0 = a - x8 * b` |
| `a * 8` | `lsl x0, x0, #3` | shift left by 3 bits (= multiply by $2^3$) |
| `a * x + b` | `madd x0, x1, x0, x2` | `x0 = x1 * x0 + x2` (multiply-add) |
| `a + b` (`int`) | `add w0, w1, w0` | the same with 32-bit registers |
| `a / b` (`unsigned`) | `udiv x0, x0, x1` | unsigned division |

</div>

* Most instructions have the form `op dest, src1, src2`: <font color="blue">the destination comes first</font>, then the operands
* `#3` is a constant (_immediate value_) written in the instruction itself
* `x8` is a register the compiler chose for a temporary value
* Whether a value is 32 or 64 bits is not in the instruction name but in the register names (`w` or `x`)
* Signed and unsigned values are the same bits in the registers; where the result differs (e.g., division), there are separate instructions (`sdiv` and `udiv`)
* The compiler replaces expensive operations with cheaper ones when it can (`* 8` by a shift)
* `big_const` shows how a large constant is built: an instruction is 32 bits long and cannot hold a 64-bit constant, so it is put together 16 bits at a time (`mov`, then `movk` = "move keep", which overwrites 16 bits and keeps the rest)

# 2. Floating point arithmetic

In [ ]:
%%writefile farith.c
double fadd_(double a, double b) { return a + b; }
double fmul_(double a, double b) { return a * b; }
double fdiv_(double a, double b) { return a / b; }
double faxpb(double a, double x, double b) { return a * x + b; }
float faxpb_f(float a, float x, float b) { return a * x + b; }
double half(double x) { return x * 0.5; }
double to_double(long a) { return a; }
long to_long(double x) { return x; }

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S farith.c
grep -vE '^\s+\.|^\s*//|^\.Lfunc' farith.s

* You will find

<div>

| C | assembly | meaning |
|---|----------|---------|
| `a + b` | `fadd d0, d0, d1` | floating point add |
| `a * b` | `fmul d0, d0, d1` | multiply |
| `a / b` | `fdiv d0, d0, d1` | divide |
| `a * x + b` | `fmadd d0, d0, d1, d2` | `d0 = d0 * d1 + d2` (_fused multiply-add_, FMA) |
| `a * x + b` (`float`) | `fmadd s0, s0, s1, s2` | the same with 32-bit registers |
| `x * 0.5` | `fmov d1, #0.5` + `fmul d0, d0, d1` | a constant is first put into a register |
| `(double)a` | `scvtf d0, x0` | signed integer to floating point |
| `(long)x` | `fcvtzs x0, d0` | floating point to signed integer, rounding toward zero |

</div>

* Floating point instructions start with `f` and use `d` (`double`) or `s` (`float`) registers
* `fmadd` computes a multiplication and an addition in one instruction, rounding only once at the end; it is the most important instruction for numerical computation, and we will see a lot of it in this course

# 3. Summary
* `op dest, src1, src2`: the destination comes first
* integer instructions (`add`, `sub`, `mul`, `sdiv`, `madd`, `lsl`, ...) use `x`/`w` registers; floating point instructions (`fadd`, `fmul`, `fdiv`, `fmadd`, ...) use `d`/`s` registers
* the register names tell the size of the values (`x`/`d` = 64 bits, `w`/`s` = 32 bits)